In [0]:
import os, sys

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from src.transforms.common import clean_strings, cast_columns, guard_unique
from src.transforms.scd2 import apply_scd2

In [0]:
BRONZE = "transit.bronze.gtfs_stops"
TARGET = "transit.silver.stops_history"
KEYS = ["stop_id"]
TRACKED = ["stop_name", "stop_lat", "stop_lon", "wheelchair_boarding"]
COORD_DP = 5
TYPES = {"stop_lat": "DOUBLE", "stop_lon": "DOUBLE", "wheelchair_boarding": "INT"}

versions = sorted(r[0] for r in spark.table(BRONZE).select("_feed_version").distinct().collect())
print("feed versions:", versions)

In [0]:
def snapshot(version):
    """One typed, trimmed, deduplicated row per stop for a single feed version."""
    raw = (spark.table(BRONZE).filter(F.col("_feed_version") == version)
           .select("stop_id", *TRACKED, "_ingested_at"))
    typed, fails, _ = cast_columns(clean_strings(raw), TYPES)
    assert sum(fails.values()) == 0, f"{version}: values that would silently become null: {fails}"
    unique, _ = guard_unique(typed, KEYS)
    return (unique
            .withColumn("stop_lat", F.round("stop_lat", COORD_DP))
            .withColumn("stop_lon", F.round("stop_lon", COORD_DP))
            .select(*KEYS, *TRACKED))

In [0]:
STAGE = "transit.silver._stops_history_stage"

dim = None
for i, v in enumerate(versions):
    dim = apply_scd2(dim, snapshot(v), KEYS, TRACKED, effective_ts=v)

    # Save and read back so Spark drops the plan behind this table.
    # Alternating two temp tables avoids overwriting a table we are still reading.
    stage = f"{STAGE}_{i % 2}"
    (dim.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(stage))
    dim = spark.table(stage)
    print(f"{v}: {dim.count():,} rows")

(dim.withColumn("_silver_loaded_at", F.current_timestamp())
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(TARGET))

spark.sql(f"DROP TABLE IF EXISTS {STAGE}_0")
spark.sql(f"DROP TABLE IF EXISTS {STAGE}_1")

silver = spark.table(TARGET)
print(f"{TARGET}  {silver.count():,} rows")

In [0]:
dupes = silver.filter("is_current").groupBy("stop_id").count().filter("count > 1").count()
assert dupes == 0, f"{dupes} stops have more than one current row"

w = Window.partitionBy("stop_id").orderBy("valid_from")
bad = (silver.withColumn("prev_to", F.lag("valid_to").over(w))
             .filter("prev_to IS NOT NULL AND prev_to != valid_from").count())
assert bad == 0, f"{bad} rows with gaps or overlaps"

print("checks 1 and 2 passed")

In [0]:
latest = versions[-1]
again = apply_scd2(spark.table(TARGET).drop("_silver_loaded_at"), snapshot(latest),
                   KEYS, TRACKED, effective_ts=latest)
assert again.count() == silver.count(), "re-applying the latest snapshot created new rows"

n_latest = snapshot(latest).count()
n_current = silver.filter("is_current").count()
assert n_current == n_latest, f"current rows {n_current} != stops in {latest} ({n_latest})"

print("all checks passed")

In [0]:
with_history = silver.groupBy("stop_id").count().filter("count > 1").count()
closed = (silver.filter("NOT is_current")
          .join(silver.filter("is_current").select("stop_id"), "stop_id", "left_anti")
          .select("stop_id").distinct().count())
print(f"rows {silver.count():,} | current {n_current:,} | stops with history {with_history:,} "
      f"| stops removed from the feed {closed:,}")

(silver.filter("stop_id IN ('WML-0214-B', '70233', 'door-dwnxg-franklin')")
       .orderBy("stop_id", "valid_from").display())